# `eml-skill` — 10-minute demo tour

This notebook walks a first-time user through the four skills in [eml-skill](https://github.com/yaniv-golan/eml-skill):

1. **`/eml-lab`** — compile ordinary math into an EML tree.
2. **`/eml-check`** — numerically audit an EML tree against a named elementary function.
3. **`/eml-optimize`** — search for a *shorter* witness tree.
4. **`/eml-fit`** — recover an exact elementary law from a CSV.

**What is EML?** The binary operator `eml(a, b) = exp(a) − log(b)` (principal branch, `cmath`). The claim of [arXiv:2603.21852](https://arxiv.org/abs/2603.21852) is that every elementary function can be written as a finite tree over `{1, x, y}` leaves joined by `eml`. This repo treats that claim as a *programmable IR* and gives you verifiers, a compiler, a fitter, and a shorter-tree searcher.

All cells below shell out to the real skill CLIs via `subprocess` — nothing is pre-baked. Re-execute the notebook to reproduce every number. It is seeded so outputs are deterministic.


In [1]:
import json
import os
import pathlib
import subprocess
import sys
import textwrap

from IPython.display import Markdown, display

# Locate the repo root from the notebook's own directory.
# (docs/demo.ipynb lives one level below the repo root.)
REPO = pathlib.Path.cwd()
if (REPO / "docs" / "demo.ipynb").exists():
    pass
elif (REPO.parent / "docs" / "demo.ipynb").exists():
    REPO = REPO.parent
else:
    REPO = pathlib.Path(__file__).resolve().parent.parent if "__file__" in globals() else REPO

ENV = {**os.environ, "PYTHONPATH": str(REPO / "skills" / "_shared")}
ARTIFACTS = REPO / "docs" / "demo-assets" / "run"
ARTIFACTS.mkdir(parents=True, exist_ok=True)


def run(args, **kw):
    """Run a skill CLI from the repo root, returning CompletedProcess."""
    return subprocess.run(
        [sys.executable, *args],
        cwd=REPO,
        env=ENV,
        capture_output=True,
        text=True,
        check=False,
        **kw,
    )


print(f"Repo root: {REPO}")
print(f"Python:    {sys.version.split()[0]}")


Repo root: /private/tmp/eml-skill-session-d
Python:    3.11.4


## 1 · `/eml-lab compile-render` — `exp(x + y)` → EML tree + diagram + audit

`compile-render` lowers a sympy-parseable expression into an EML tree by substituting library witnesses, renders the tree as Mermaid, and audits the compiled tree numerically against `sympy.lambdify(..., modules="cmath")`. A single command produces five artifacts: `tree.txt`, `diagram.md`, `audit.json`, `audit.md`, `summary.md`.

The flagship `sin(sqrt(x) + cos(x))` compiles to K=1151 (too large for GitHub's Mermaid renderer) — we'll show that stat below, but render a small example first so the tree fits on-screen.

In [2]:
out_small = ARTIFACTS / "compile_exp_x_plus_y"
proc = run([
    "eml-skill/skills/eml-lab/scripts/lab.py",
    "compile-render",
    "--expr", "exp(x + y)",
    "--out-dir", str(out_small),
])
summary = json.loads(proc.stdout)
print(json.dumps(summary, indent=2))


{
  "mode": "compile-render",
  "expr": "exp(x + y)",
  "verdict": "verified",
  "K": 21,
  "depth": 9,
  "used_witnesses": [
    "add",
    "exp"
  ],
  "max_abs_diff": 1.2809491335957507e-14,
  "artifacts": {
    "tree": "tree.txt",
    "diagram": "diagram.md",
    "audit_json": "audit.json",
    "audit_md": "audit.md",
    "summary": "summary.md"
  },
  "out_dir": "/private/tmp/eml-skill-session-d/docs/demo-assets/run/compile_exp_x_plus_y"
}


In [3]:
display(Markdown((out_small / "diagram.md").read_text()))

```mermaid
---
title: EML: exp(x + y)
---
flowchart TD
  n0[eml]
  n1[eml]
  n2(("1"))
  n3[eml]
  n4[eml]
  n5[eml]
  n6(("1"))
  n7[eml]
  n8[eml]
  n9(("1"))
  n10[eml]
  n11(("1"))
  n12[eml]
  n13(("x"))
  n14(("1"))
  n12 -->|a| n13
  n12 -->|b| n14
  n10 -->|a| n11
  n10 -->|b| n12
  n8 -->|a| n9
  n8 -->|b| n10
  n15(("1"))
  n7 -->|a| n8
  n7 -->|b| n15
  n5 -->|a| n6
  n5 -->|b| n7
  n16[eml]
  n17(("y"))
  n18(("1"))
  n16 -->|a| n17
  n16 -->|b| n18
  n4 -->|a| n5
  n4 -->|b| n16
  n19(("1"))
  n3 -->|a| n4
  n3 -->|b| n19
  n1 -->|a| n2
  n1 -->|b| n3
  n20(("1"))
  n0 -->|a| n1
  n0 -->|b| n20
```


### The flagship: `sin(sqrt(x) + cos(x))`

Same command, bigger expression. We print only the summary — the Mermaid tree would exceed GitHub's ~500-node rendering limit, but the generated `diagram.md` is still valid and viewable in any Mermaid renderer that supports larger graphs.

In [4]:
out_big = ARTIFACTS / "compile_sin_sqrt_plus_cos"
proc = run([
    "eml-skill/skills/eml-lab/scripts/lab.py",
    "compile-render",
    "--expr", "sin(sqrt(x) + cos(x))",
    "--out-dir", str(out_big),
    "--domain", "positive-reals",
])
flagship = json.loads(proc.stdout)
print(json.dumps(flagship, indent=2))


{
  "mode": "compile-render",
  "expr": "sin(sqrt(x) + cos(x))",
  "verdict": "verified-with-caveats",
  "K": 1151,
  "depth": 74,
  "used_witnesses": [
    "sqrt",
    "cos",
    "add",
    "sin"
  ],
  "max_abs_diff": 5.341148072078276e-13,
  "artifacts": {
    "tree": "tree.txt",
    "diagram": "diagram.md",
    "audit_json": "audit.json",
    "audit_md": "audit.md",
    "summary": "summary.md"
  },
  "out_dir": "/private/tmp/eml-skill-session-d/docs/demo-assets/run/compile_sin_sqrt_plus_cos"
}


## 2 · `/eml-check --format blog` — audit a claimed `ln` witness

`/eml-check` takes a tree and a named claim (one of 20 primitives) and returns a structured audit: numerical agreement on interior samples plus branch-cut probes. `--format blog` renders the report as a self-contained markdown artifact — the same format used for leaderboard entries.

The `ln` witness `eml(1, eml(eml(1, x), 1))` is K=7 — short enough that the Mermaid tree renders inline on GitHub. Note the branch-cut probes on the negative real axis: `ln` has a branch cut there, and principal-branch `cmath` gets both sides of the cut right.

In [5]:
out_audit = ARTIFACTS / "audit_ln"
proc = run([
    "eml-skill/skills/eml-check/scripts/audit.py",
    "--tree", "eml(1, eml(eml(1, x), 1))",
    "--claim", "ln",
    "--out-dir", str(out_audit),
    "--format", "blog",
])
# Strip the trailing timestamp line so cell output is deterministic across re-runs.
blog_md = (out_audit / "audit.blog.md").read_text()
blog_md = "\n".join(
    line for line in blog_md.splitlines()
    if not line.startswith("_Generated by ")
).rstrip()
display(Markdown(blog_md))


# 🟡 `ln` — upper bound at K=7

**Audit verdict**: `verified` — max |diff| = 8.88178e-16 on `positive-reals` (70 interior samples, tolerance 1e-10).

## Tree

```mermaid
graph TD
    n1["eml"]
    n2(("1"))
    n3["eml"]
    n4["eml"]
    n5(("1"))
    n6(("x"))
    n4 -->|a| n5
    n4 -->|b| n6
    n7(("1"))
    n3 -->|a| n4
    n3 -->|b| n7
    n1 -->|a| n2
    n1 -->|b| n3
```

- **K (RPN tokens)**: 7  · **depth**: 3  · **leaves**: 1×3, x×1, y×0

## K context

| source | K | notes |
|--------|--:|-------|
| our `WITNESSES` | 7 | upper bound (shorter witness may exist) |
| paper (arXiv:2603.21852, Table 4) | — | not machine-readable; see `docs/internal/kvalues.md` |
| proof-engine | — | see proof page in provenance below |

## Provenance

- **Proof page**: [https://yaniv-golan.github.io/proof-engine/proofs/eml-triple-nesting-recovers-ln-x/](https://yaniv-golan.github.io/proof-engine/proofs/eml-triple-nesting-recovers-ln-x/)
- **Library note**:

  > triple-nesting identity; principal-branch cmath

## Branch-cut probes

| locus | sample | max \|diff\| | passed |
|-------|--------|--------------:|:------:|
| `neg-real-axis` | `(-0.1+1e-06j)` | 0 | ✅ |
| `neg-real-axis` | `(-0.1-1e-06j)` | 0 | ✅ |
| `neg-real-axis` | `(-1+1e-06j)` | 4.44503e-17 | ✅ |
| `neg-real-axis` | `(-1-1e-06j)` | 4.44503e-17 | ✅ |
| `neg-real-axis` | `(-5+1e-06j)` | 0 | ✅ |
| `neg-real-axis` | `(-5-1e-06j)` | 0 | ✅ |

---

## 3 · `/eml-optimize search` — find a shorter witness for `sub`

Beam search enumerates EML trees bottom-up with meet-in-the-middle hashing, backward goal propagation, and optional library-witness seeding. For `sub(x, y) = x − y`, it rediscovers a K=11 tree in under a second — a K=2 improvement over the library entry.

> **"Found" ≠ "minimal"**: the search reports the shortest K *it found* within > the cap/budget. Exhaustive minimality is `/eml-check`'s `minimality.py`.

In [6]:
proc = run([
    "eml-skill/skills/eml-optimize/scripts/optimize.py",
    "search",
    "--target", "sub",
    "--max-k", "13",
])
# stdout is markdown already — strip the "Time" line so output is deterministic.
lines = [line for line in proc.stdout.splitlines() if not line.lstrip().startswith("- **Time**")]
display(Markdown("\n".join(lines)))


# Beam search: target `sub`

- **Found**: True
- **K**: 11
- **RPN**: `1 1 x E 1 E E y 1 E E`
- **Depth**: 4
- **Leaves**: {'1': 4, 'x': 1, 'y': 1}
- **Candidates evaluated**: 471
- **Stopped**: generalized-targeted-hit

## Per-K unique function counts
- K=1: 3
- K=3: 9
- K=5: 54
- K=7: 405

## Equivalence re-gate
- passed: True
- samples: 1024
- max_abs_diff: 9.930e-16

## 4 · `/eml-fit` — recover an exact law from a CSV

Deterministic library-first regression: fit a CSV against the witness library and emit a machine-checkable JSON verdict. Affine mode fits `y ≈ a·w(x) + b` and snaps `a`, `b` to named constants (`π`, `e`, `ln(2)`, Catalan's G, ...).

We generate 30 samples of `y = π·sin(x) + 1` (no noise) and ask `/eml-fit` to recover the law. It should return `verdict: matched`, `best.name: sin`, `a → π`, `b → 1`.

In [7]:
import csv
import math

csv_path = ARTIFACTS / "pi_sin_plus_one.csv"
with csv_path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["x", "y"])
    for i in range(30):
        x = 0.05 + 0.2 * i          # deterministic grid; no RNG
        y = math.pi * math.sin(x) + 1.0
        w.writerow([f"{x:.17g}", f"{y:.17g}"])

proc = run([
    "eml-skill/skills/eml-fit/scripts/fit.py",
    "--csv", str(csv_path),
    "--affine",
    "--top-k", "3",
])
fit = json.loads(proc.stdout)
best = fit["best"]
print(f"verdict:          {fit['verdict']}")
print(f"best witness:     {best['name']}")
print(f"a = {best['a']['real']:.15f}  →  snapped to {best['a_snapped']!r}")
print(f"b = {best['b']['real']:.15f}  →  snapped to {best['b_snapped']!r}")
print(f"max_abs_residual: {best['max_abs_residual']:.3e}")


verdict:          matched
best witness:     sin
a = 3.141592653589793  →  snapped to 'pi'
b = 1.000000000000000  →  snapped to '1'
max_abs_residual: 8.882e-16


## 5 · What's next

- **Paper**: [arXiv:2603.21852](https://arxiv.org/abs/2603.21852) — the EML paper.
- **Proof engine**: [yaniv-golan.github.io/proof-engine](https://yaniv-golan.github.io/proof-engine/) — human-readable proofs per witness.
- **Witness library**: `eml-skill/skills/_shared/eml_core/witnesses.py` — every primitive, its tree, its proof URL. Append-only.
- **Public leaderboard**: *coming soon* in Phase 2 (`docs/leaderboard.md`). Until then, the project README holds the compact inventory table.
- **Skill docs**: `eml-skill/skills/eml-check/SKILL.md`, `eml-skill/skills/eml-lab/SKILL.md`, `eml-skill/skills/eml-fit/SKILL.md`, `eml-skill/skills/eml-optimize/SKILL.md`.

Open questions and iteration notes live in each `SKILL.md` under "Gotchas" / "Non-goals". Corrections, shorter witnesses, and additional primitives are welcome as PRs that append to `WITNESSES` with a proof URL and a test.
